In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, glob, importlib, time
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0   = 20260803
IDX_S   = 1
NB_BOOT = 4000

# ── 창 (R = index 100 · 360Hz) — Q7-K 승계
STT_SEG  = (130, 215)
STT_SAFE = (130, 175)
INTRUDE_LIM = STT_SEG[1] - 100
SEGS = {"p_full": (0, 85), "p_early": (0, 32), "p_late": (53, 85),
        "stt": STT_SEG, "stt_safe": STT_SAFE}

# ── 사전등록 상수. **이 아래 어느 셀에서도 다시 고르지 않는다.**
KS         = (8, 16)
LB_K       = 16             # 정합 두 번째 축 = 국소 기저선 (f2_16 과 짝)
MIN_S_TPL  = 20
K_FOLD     = 5              # R22
N_REPEAT   = 3
N_SHUF     = 20             # R26 ② — null 의 셔플 오차를 CI 에 전파
LB_BANDS   = (0.10, 0.06, 0.04, 0.02)   # ★ **국소 기저선만** 대역. pre 는 항상 정확
POST_BAND  = 0.06           # M4 3축의 post 대역 (사전등록)

# ★★ 이번 실험의 핵심 상수 변경 —— 레거시 감사
#   조건부 일치도는 **칸마다 추정치를 만들지 않는다.** 이긴 쌍/전체 쌍을 누적할 뿐이라
#   **S 1 · N 1 칸도 유효한 쌍 1개**를 기여한다. `≥3` 은 Q7-F 의 `matched_auc` 에서
#   물려받았고 정당화된 적이 없다. 개체당 `pre` 고유값이 193개인 조건에서 대부분의
#   칸을 죽인다. 편의가 아니라 **분산만** 늘고, 그건 개체 부트스트랩이 이미 잡는다.
MIN_CELL_S, MIN_CELL_N = 1, 1
AUDIT_CELL = (3, 2, 1)      # 【M-0】 이 회복량을 실측한다
MIN_REC_S, MIN_REC_PAIR = 15, 100
MIN_REC = 10

EXACT_TIE   = 0.505         # M1 — pre 가 값 단위로 같으니 f1 은 동점뿐
F2_COLLAPSE = 0.56          # ★ M2 전제 — 이 아래여야 「조기성을 통제했다」
BONF3       = 0.05 / 3 / 2  # 1차 가족 {M3a · M3b · M4}
ISO_HI, ISO_LO = 0.7, 0.3

CONFIG = dict(
    exp="quest46_q7m_recover_power", quest="ailab-2026-0046", step="svdb-recover-power",
    parent_exp=["quest46_q7k_relative_match", "ailab-2026-0061"],
    purpose=("Q7-K 는 두 축을 **같은 폭 대역**으로 갈라 정합했고 어느 쪽도 못 죽였다"
             "(f1ₘ 0.5526 · f2ₘ 0.5445). 원인은 둘이다 — ① **정확 pre 를 팔았다**: "
             "f1 은 pre 의 결정론적 단조함수라 잔여 1.7샘플에도 순위가 산다(R27 ①). "
             "② **표본을 버렸다**: 19/59 개체 · S 의 25%. 이번엔 **pre 를 항상 정확 "
             "정합으로 고정**하고 **국소 기저선만 대역**으로 묶는다 — `pre` 를 고정하면 "
             "`f2 = 1 − pre/b` 는 **`b` 의 순증가 함수**라 `b` 를 묶는 것이 `f2` 를 "
             "묶는 것과 정확히 같다. 동시에 **`MIN_CELL` 을 3 → 1** 로 내려 버리던 "
             "칸을 되찾는다(레거시 상수 감사). 그 조건에서 **차이를 주지표로** 묻는다"),
    dataset="SVDB 전수 · svdb_data5.npz + Q7-B 예측 캐시(라벨·매핑용)",
    lb_bands=list(LB_BANDS), ks=list(KS), lb_k=LB_K, n_shuffle=N_SHUF,
    min_cell=[MIN_CELL_S, MIN_CELL_N], audit_cell=list(AUDIT_CELL),
    windows={k: list(v) for k, v in SEGS.items()}, intrude_lim=INTRUDE_LIM,
    post_band=POST_BAND,
    predictions={
        "M0": "(관문 아님) ★ **레거시 상수 감사** — MIN_CELL 3→2→1 로 회복되는 "
              "개체 수·남은 S 비율. Q7-I‴(20/200)·Q7-K(15/100) 문턱도 한 표에",
        "M1": f"정확 pre 정합에서 f1ₘ CI 상한 < {EXACT_TIE} — 동점뿐이라는 항등식",
        "M2": f"★ **전제** — f2_16ₘ CI 상한 < {F2_COLLAPSE}. 두 축을 **처음으로 "
              "동시에** 통제한다(Q7-K 는 0.5445 미결 · Q7-I‴ 은 0.6688). "
              "미지지면 아래 **수준**을 봉인한다(R24-b)",
        "M3a": "★ **주지표는 차이** — (P_late 초과) − (STT 초과) (Bonferroni 3). "
               "0 이면 **「P 파 고유의 신호는 없다」**가 확정된다(Q7-F·Q7-K 에 이어 3번째)",
        "M3b": "★ (STT 초과) − (f3 초과) CI 하한 > 0 (Bonferroni 3) — "
               "형태가 보상성 휴지를 이기는가",
        "M4": "★ **3축**(정확 pre × 국소기저선 × post) 에서 STT 초과 CI 하한 > 0 "
              "(Bonferroni 3). 사라지면 STT 는 **재분극 rate hysteresis** = 리듬의 "
              "다른 얼굴이다. Q7-K L8 은 개체 6개라 판정 불가였다",
        "M5": "f3_glob(분모를 전역 중앙으로) 은 기저선 대역에 **반응하지 않는다** — "
              "반응하면 「f3 붕괴는 국소 기저선 때문」이라는 해석이 틀린 것이다",
        "M6": "(관문 아님) 층 구성 · 선택 편향 · leave-one-out · 상쇄 검증 · "
              "층화 vs 매크로 간극 · 계수 부호 안정성"},
    caveat=("**M2 가 지지가 아니면 아래 「수준」을 인용하지 않는다.** ★ 단 **같은 조건 "
            "두 팔의 차이**(M3a·M3b)와 **대역별 반응 곡선**은 공통 오염이 상쇄되므로 "
            "**봉인 아래에서도 읽는다**(R27 ②) — 그래서 이번엔 처음부터 **차이를 "
            "1차 지표로** 사전등록한다. 유효 대역은 **규칙이 고른다**(가능 개체 ≥ "
            f"{MIN_REC} 중 가장 좁은 것). 학습이 낀 팔은 전부 **자기 라벨셔플 null 위 "
            "초과분**으로 읽고 null 의 셔플 오차를 CI 에 전파한다(R26 ②). 순수 특징의 "
            "영분포는 0.5 다. 개체 내부 로지스틱·템플릿은 **라벨을 쓰는 상한**이지 "
            "배포 모형이 아니다. ★ **실험 간 세로 비교 금지** — Q7-I‴(22개체)와 "
            "Q7-K(25개체)는 개체 문턱(20/200 vs 15/100)이 달라 초과분을 나란히 놓으면 "
            "안 된다. 이 노트북 안의 **정확 pre 단독 행이 그 재현**이고, 모든 비교는 "
            "**같은 실행·같은 코호트**에서만 한다. ★ 정합은 코호트를 고른다 — 층 구성이 "
            "0 에 가까운 층이 있으면 **결론의 적용 범위를 그 층 밖으로 넓히지 않는다**. "
            "학습 0회 · GPU 불필요 · 예상 20~40분(셔플 20회가 지배)"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7m_recover_power", CONFIG, project=PROJECT)
run.log("설정 ✅ 정확 pre 고정 + 국소기저선만 대역 · MIN_CELL " + str(MIN_CELL_S))

In [ ]:
# CELL 2 — 【M-0a】 자산 · 매핑 (Q7-D/E/F/H/I/I″/I‴/K 와 동일 규약 — fallback 없음 R16)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # ⛔ fallback 없음 (R16)
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다 — 연속 번호로 대체하지 않는다")
labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)
assert len(np.unique(REC)) == len(labels) and not (set(REC.tolist()) - set(recs))

d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"길이 불일치 {int(keep.sum())} vs {len(Y)}"
PRE  = np.asarray(d5["pre_rr"])[keep].astype(float)
POST = np.asarray(d5["post_rr"])[keep].astype(float)
BEAT = np.asarray(d5["beat"])[keep]
WIN = {k: np.ascontiguousarray(BEAT[:, :, a:b]).astype("float32")
       for k, (a, b) in SEGS.items()}
del BEAT
ALLR = [int(r) for r in np.unique(REC)]
run.log("\n" + "=" * 100)
run.log("【M-0a】 자산 · 매핑")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(ALLR)}개 · "
        "창 " + " · ".join(f"{k}{v}" for k, v in SEGS.items()))
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【M-A】 특징·점수 (승계 + **f3_glob 신설**)
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

def local_base(pre_v, k):
    n = len(pre_v); out = np.empty(n); med = float(np.median(pre_v))
    for i in range(n):
        a = max(0, i - k)
        out[i] = med if i - a < 3 else float(np.median(pre_v[a:i]))
    return out

def rhythm_feats(pre_v, post_v, ks):
    """Q7-K 승계 + **f3_glob**: `f3` 의 분모를 국소 기저선 → **전역 중앙**으로.

    Q7-K 에서 `f3` 는 국소 기저선을 맞출수록 단조 붕괴했다(0.5815 → 0.4886).
    해석은 「`f3` 의 분모가 국소 기저선이라 그걸 맞추면 남는 게 `post` 뿐」이었다.
    맞다면 **분모를 전역 중앙으로 바꾼 `f3_glob` 은 기저선 대역에 반응하지 않아야**
    한다 — 반응하면 그 해석이 틀린 것이다. **M5 는 그 반증 시험**이다."""
    med = float(np.median(pre_v)); F, NM = [], []
    F.append(med - pre_v); NM.append("f1")
    b_first = None; b_lb = None
    for k in ks:
        b = local_base(pre_v, k)
        if b_first is None:
            b_first = b
        if k == LB_K:
            b_lb = b
        F.append(1.0 - pre_v / np.maximum(b, 1e-9)); NM.append(f"f2_{k}")
    if b_lb is None:
        raise AssetError(f"LB_K={LB_K} 가 KS={ks} 에 없다 — 정합 키를 만들 수 없다")
    F.append(1.0 - (pre_v + post_v) / np.maximum(2.0 * b_first, 1e-9)); NM.append("f3")
    F.append(1.0 - (pre_v + post_v) / (2.0 * max(med, 1e-9)));          NM.append("f3_glob")
    cv = np.empty(len(pre_v))
    for i in range(len(pre_v)):
        a = max(0, i - ks[0]); w = pre_v[a:i] if i - a >= 3 else pre_v[:3]
        cv[i] = float(np.std(w) / max(np.mean(w), 1e-9))
    F.append(cv); NM.append("f4")
    F.append(np.r_[0.0, F[1][:-1]]); NM.append("f5")
    F.append(1.0 - b_lb / max(med, 1e-9)); NM.append("f6")
    return np.stack(F, axis=1), NM, b_lb

def dist(B, ref):
    d = B - ref[None]
    return np.sqrt((d * d).sum(axis=(1, 2)))

def cv_logit(X, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            mu = X[tr].mean(0); sd = X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=2000, C=1.0)
            lr.fit((X[tr] - mu) / sd, tt[tr].astype(int))
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def two_template_cv(B, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            medN = np.median(B[tr & ~tt], axis=0); medS = np.median(B[tr & tt], axis=0)
            sc[te] = dist(B[te], medN) - dist(B[te], medS)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def run_struct(t_):
    prev_ = np.r_[False, t_[:-1]]; next_ = np.r_[t_[1:], False]
    iso = t_ & ~prev_ & ~next_
    rl = mx = 0
    for v in t_:
        rl = rl + 1 if v else 0
        mx = max(mx, rl)
    return float(iso.sum() / max(t_.sum(), 1)), int(mx)

def build_scores(X, NAMES, tt, MORPH, seed):
    """모든 팔의 비트별 점수. **라벨셔플 null 도 같은 함수로** 만든다(R26 ②)."""
    S = {nm_: X[:, j] for j, nm_ in enumerate(NAMES)}
    i3, i4, i5 = NAMES.index("f3"), NAMES.index("f4"), NAMES.index("f5")
    arms = {"lr_all": X, "lr_norr": X[:, [i3, i4, i5]], "lr_f1": X[:, [NAMES.index("f1")]]}
    for nm_, XX in arms.items():
        sc_ = cv_logit(XX, tt, K_FOLD, seed, N_REPEAT)
        if sc_ is None:
            return None
        S[nm_] = sc_
    for nm_, B_ in MORPH.items():
        st = two_template_cv(B_, tt, K_FOLD, seed, N_REPEAT)
        S[nm_] = st if st is not None else np.full(len(tt), np.nan)
    return S

run.log("\n" + "=" * 100)
run.log("【M-A】 특징·점수 계산 (+ f3_glob)")
run.log("=" * 100)
T0 = time.time()
SCORES, META, SKIP = {}, {}, []
for r in ALLR:
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    if int(tt.sum()) < MIN_S_TPL or int((~tt).sum()) < MIN_S_TPL:
        SKIP.append((int(r), f"S {int(tt.sum())} · N {int((~tt).sum())}")); continue
    pre_m, post_m = PRE[mm], POST[mm]
    X, NAMES, lb_m = rhythm_feats(pre_m, post_m, KS)
    MORPH = {k: WIN[k][mm] for k in SEGS}
    S = build_scores(X, NAMES, tt, MORPH, SEED0)
    if S is None:
        SKIP.append((int(r), "겹 안 클래스 부족")); continue
    iso_f, mx_run = run_struct(tt)
    SCORES[int(r)] = S
    META[int(r)] = dict(tt=tt, pre=pre_m, post=post_m, lb=lb_m, X=X, MORPH=MORPH,
                        pos=int(tt.sum()), prev=float(tt.mean()),
                        iso_frac=iso_f, max_run=mx_run)
RS = sorted(SCORES)
run.log(f"  채점 {len(RS)}개체 · 제외 {len(SKIP)}개체 · {time.time()-T0:.0f}초")
if len(RS) < 10:
    raise AssetError("채점된 개체가 너무 적다")
ARMS = (["f1", "f2_8", "f2_16", "f3", "f3_glob", "f4", "f5", "f6",
         "lr_f1", "lr_all", "lr_norr"] + list(SEGS))
NAMES_G = NAMES
RAW = {a: np.array([roc_auc_score(META[r]["tt"].astype(int), SCORES[r][a])
                    if np.isfinite(SCORES[r][a]).all() else np.nan for r in RS]) for a in ARMS}
run.log("  무정합 매크로 — " + " · ".join(f"{a} {np.nanmean(RAW[a]):.4f}" for a in
        ("f1", "f2_16", "f3", "f3_glob", "stt", "p_late", "lr_all")))
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【M-0】 ★ 레거시 상수 감사 — MIN_CELL 을 낮추면 얼마나 되찾나
# 조건부 일치도는 칸마다 추정치를 만들지 않는다. 이긴 쌍/전체 쌍을 누적할 뿐이라
# **S 1 · N 1 칸도 유효한 쌍 1개**다. `≥3` 은 Q7-F 에서 물려받은 값이고 정당화된
# 적이 없다 — 편의가 아니라 **분산만** 늘리며 그건 개체 부트스트랩이 잡는다.
def exact_key(pre_v):
    _, ia = np.unique(np.round(pre_v, 6), return_inverse=True)
    return ia.astype(np.int64)

def joint_key(pre_v, lb_v, bw, post_v=None, pbw=None):
    """★ `pre` 는 **항상 정확**(값 단위), 국소 기저선만 대역.

    `pre` 를 고정하면 `f2 = 1 − pre/b` 는 **`b` 의 순증가 함수**다 — `b` 를 대역으로
    묶는 것이 `f2` 를 묶는 것과 **정확히 같다**. Q7-K 는 두 축을 같은 폭 대역으로
    갈랐고 그래서 `f1` 을 잃었다(R27 ①). `pbw` 를 주면 `post` 축까지 3축(M4)."""
    k = exact_key(pre_v)
    if bw is not None:
        b = np.floor(lb_v / max(bw, 1e-9)).astype(np.int64); b = b - b.min()
        k = k * (int(b.max()) + 1) + b
    if pbw is not None and post_v is not None:
        p = np.floor(post_v / max(pbw, 1e-9)).astype(np.int64); p = p - p.min()
        k = k * (int(p.max()) + 1) + p
    return k

def cell_auc(sc, tt, key, min_s, min_n, pre_v=None, f2_v=None):
    """`key` 가 같은 비트끼리만 쌍을 센다. **평가만 제한**(승계).
    반환 (조건부 AUROC, 남은 S, 쌍, 칸수, 짝단위 f1 잔여, 짝단위 f2 잔여)."""
    uq, inv = np.unique(key, return_inverse=True)
    num = den = 0.0; ks_ = nc_ = 0; g1 = g2 = 0.0
    for j in range(len(uq)):
        m = inv == j
        s_, n_ = sc[m & tt], sc[m & ~tt]
        if len(s_) < min_s or len(n_) < min_n:
            continue
        ks_ += len(s_); nc_ += 1
        gt = float((s_[:, None] > n_[None, :]).sum())
        eq = float((s_[:, None] == n_[None, :]).sum())
        num += gt + 0.5 * eq; den += float(len(s_) * len(n_))
        if pre_v is not None:
            a_, b_ = pre_v[m & tt], pre_v[m & ~tt]
            g1 += float(np.abs(a_[:, None] - b_[None, :]).sum())
            a2, b2 = f2_v[m & tt], f2_v[m & ~tt]
            g2 += float(np.abs(a2[:, None] - b2[None, :]).sum())
    if den < 1:
        return float("nan"), ks_, 0.0, nc_, float("nan"), float("nan")
    if pre_v is None:
        return num / den, ks_, den, nc_, float("nan"), float("nan")
    return (num / den, ks_, den, nc_,
            (g1 / den) / max(float(np.median(pre_v)), 1e-9), g2 / den)

run.log("\n" + "=" * 100)
run.log("【M-0】 ★ 레거시 상수 감사 — MIN_CELL 을 낮추면 표본이 얼마나 돌아오나")
run.log("=" * 100)
AUD = {}
for mc in AUDIT_CELL:
    for lab, bw_f in (("정확pre", None), ("+기저선 0.04", 0.04)):
        ok = 0; sf = []; pr_all = []
        for r in RS:
            tt = META[r]["tt"]; pre_m = META[r]["pre"]
            bw = None if bw_f is None else bw_f * float(np.median(pre_m))
            key = joint_key(pre_m, META[r]["lb"], bw)
            _, ks_, pr_, _, _, _ = cell_auc(SCORES[r]["f1"], tt, key, mc, mc)
            if ks_ >= MIN_REC_S and pr_ >= MIN_REC_PAIR:
                ok += 1; sf.append(ks_ / max(META[r]["pos"], 1)); pr_all.append(pr_)
        AUD[f"{mc}|{lab}"] = dict(n_ok=ok, s_frac=float(np.median(sf)) if sf else None,
                                  pairs=float(np.median(pr_all)) if pr_all else None)
        star = "  ★ 이번 실험" if (mc == MIN_CELL_S and bw_f == 0.04) else ""
        run.log(f"  MIN_CELL={mc} · {lab:<12} — 개체 **{ok:>2}/{len(RS)}** · "
                f"남은 S 중앙 **{np.median(sf) if sf else float('nan'):.3f}** · "
                f"쌍 중앙 {np.median(pr_all) if pr_all else float('nan'):.0f}{star}")
run.log("\n  개체 문턱 비교 — Q7-I‴ 은 (20 S, 200 쌍) · Q7-K 는 (15, 100) 이라")
run.log(f"  **같은 조건인데 개체 수가 달랐다**(22 vs 25). 이 노트북은 ({MIN_REC_S}, {MIN_REC_PAIR}) "
        "하나로 고정하고, **모든 비교를 이 실행 안에서만** 한다")
CONFIG["audit_cell"] = AUD
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【M-B】 ★ 정확 pre × 국소기저선 대역 — 곡선
run.log("\n" + "=" * 100)
run.log("【M-B】 정확 pre 고정 + 국소기저선 대역 — 짝 단위 잔여를 f1·f2 둘 다")
run.log("=" * 100)
CURVE = {}
for bw_f in (None,) + LB_BANDS:
    M = {a: np.full(len(RS), np.nan) for a in ARMS}
    ok = np.zeros(len(RS), bool)
    g1 = np.full(len(RS), np.nan); g2 = np.full(len(RS), np.nan)
    sf = np.full(len(RS), np.nan); nc = np.full(len(RS), np.nan)
    for i, r in enumerate(RS):
        tt = META[r]["tt"]; pre_m = META[r]["pre"]
        f2_m = SCORES[r][f"f2_{LB_K}"]
        bw = None if bw_f is None else bw_f * float(np.median(pre_m))
        key = joint_key(pre_m, META[r]["lb"], bw)
        for a in ARMS:
            sc_ = SCORES[r][a]
            if not np.isfinite(sc_).all():
                continue
            v, ks_, pr_, nc_, r1, r2 = cell_auc(
                sc_, tt, key, MIN_CELL_S, MIN_CELL_N,
                pre_v=pre_m if a == "f1" else None, f2_v=f2_m if a == "f1" else None)
            M[a][i] = v
            if a == "f1":
                ok[i] = bool(ks_ >= MIN_REC_S and pr_ >= MIN_REC_PAIR)
                g1[i] = r1; g2[i] = r2; sf[i] = ks_ / max(META[r]["pos"], 1); nc[i] = nc_
    CURVE[bw_f] = dict(M=M, ok=ok, g1=g1, g2=g2, sf=sf, nc=nc)
    lab = "정확pre만" if bw_f is None else f"{bw_f:.2f}"
    run.log(f"  기저선 대역 {lab:>8} — 가능 **{int(ok.sum()):>2}/{len(RS)}** · "
            f"남은 S {np.nanmedian(sf[ok]) if ok.any() else float('nan'):.3f} · "
            f"짝잔여 f1 **{np.nanmedian(g1[ok]) if ok.any() else float('nan'):.6f}** · "
            f"**f2 {np.nanmedian(g2[ok]) if ok.any() else float('nan'):.4f}** | "
            + " · ".join(f"{a} {np.nanmean(M[a][ok]) if ok.any() else float('nan'):.4f}"
                         for a in ("f1", "f2_16", "f3", "f3_glob", "stt", "p_late")))

elig = [b for b in LB_BANDS if int(CURVE[b]["ok"].sum()) >= MIN_REC]
if not elig:
    raise AssetError(f"가능 개체 ≥ {MIN_REC} 인 기저선 대역이 없다 — 여기서 멈춘다")
BW = min(elig)
JOK = CURVE[BW]["ok"]; JM = CURVE[BW]["M"]; NJ = int(JOK.sum())
run.log(f"\n  ★ 유효 대역(규칙: 가능 ≥ {MIN_REC} 중 가장 좁은 것) = **{BW:.2f}** · 개체 {NJ}")
run.log("    (첫 줄 = Q7-I‴ 재현. `f1` 이 0.5 인데 `f2` 가 살아 있으면 K3 재확인이다)")
CONFIG["curve"] = {("exact_only" if b is None else f"{b:.2f}"): dict(
    n_ok=int(CURVE[b]["ok"].sum()),
    gap_f1=float(np.nanmedian(CURVE[b]["g1"][CURVE[b]["ok"]])) if CURVE[b]["ok"].any() else None,
    gap_f2=float(np.nanmedian(CURVE[b]["g2"][CURVE[b]["ok"]])) if CURVE[b]["ok"].any() else None,
    s_frac=float(np.nanmedian(CURVE[b]["sf"][CURVE[b]["ok"]])) if CURVE[b]["ok"].any() else None,
    macro={a: float(np.nanmean(CURVE[b]["M"][a][CURVE[b]["ok"]])) if CURVE[b]["ok"].any()
           else None for a in ARMS}) for b in CURVE}
CONFIG["band_selected"] = BW
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【M-C】 라벨셔플 null (셔플 20회 · SE 를 CI 에 전파 · R26 ②)
run.log("\n" + "=" * 100)
run.log(f"【M-C】 라벨셔플 null (셔플 {N_SHUF}회 · 대역 {BW:.2f} · 개체 {NJ})")
run.log("=" * 100)
T1 = time.time()
NULL = {a: np.full(len(RS), np.nan) for a in ARMS}
NSE  = {a: np.full(len(RS), np.nan) for a in ARMS}
for i, r in enumerate(RS):
    if not JOK[i]:
        continue
    tt = META[r]["tt"]; pre_m = META[r]["pre"]
    key = joint_key(pre_m, META[r]["lb"], BW * float(np.median(pre_m)))
    acc = {a: [] for a in ARMS}
    for s_ in range(N_SHUF):
        rng = np.random.RandomState(SEED0 + 7919 * (s_ + 1) + int(r))
        ts = rng.permutation(tt)                       # 유병률 보존
        Ss = build_scores(META[r]["X"], NAMES_G, ts, META[r]["MORPH"], SEED0 + 31 * (s_ + 1))
        if Ss is None:
            continue
        for a in ARMS:
            sc_ = Ss[a]
            if not np.isfinite(sc_).all():
                continue
            v, _, pr_, _, _, _ = cell_auc(sc_, ts, key, MIN_CELL_S, MIN_CELL_N)
            if pr_ >= 1:
                acc[a].append(v)
    for a in ARMS:
        if len(acc[a]) >= 2:
            v_ = np.asarray(acc[a], float)
            NULL[a][i] = float(v_.mean()); NSE[a][i] = float(v_.std(ddof=1) / np.sqrt(len(v_)))
        elif acc[a]:
            NULL[a][i] = float(acc[a][0]); NSE[a][i] = float("nan")
run.log(f"  ({time.time()-T1:.0f}초)  팔별 — 실측 vs **셔플 null(±SE)** vs 초과분")
for a in ARMS:
    m_ = np.nanmean(JM[a][JOK]); n_ = np.nanmean(NULL[a][JOK]); se_ = np.nanmean(NSE[a][JOK])
    star = "  ★" if a in ("stt", "stt_safe", "p_late", "f3", "f3_glob") else ""
    run.log(f"    {a:<9} {m_:.4f}  null {n_:.4f} ±{se_:.4f}  **초과 {m_-n_:+.4f}**{star}")
CONFIG["null"] = {a: float(np.nanmean(NULL[a][JOK])) for a in ARMS}
CONFIG["null_se"] = {a: float(np.nanmean(NSE[a][JOK])) for a in ARMS}
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【M-D】 관문 M1·M2·M3a·M3b·M5
def boot_diff(a, b, seed, nb=NB_BOOT, mask=None, const=None, q=2.5):
    a = np.asarray(a, float)
    d = (a - const) if const is not None else (a - np.asarray(b, float))
    if mask is not None:
        d = d[mask]
    d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)])
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

def excess(arm, mask=None):
    """초과분 벡터와 그 셔플 SE. 순수 특징은 null 이 0.5 지만 **실측 null 을 쓴다**."""
    return JM[arm] - NULL[arm], NSE[arm]

def boot_excess_diff(a1, a2, seed, mask, q=2.5, nb=NB_BOOT):
    """두 팔의 **초과분 차이**. 각 팔의 null 오차를 함께 흔든다(R26 ②).
    a2=None 이면 a1 의 초과분 자체."""
    d1, s1 = excess(a1)
    if a2 is None:
        d, se = d1, s1
    else:
        d2, s2 = excess(a2)
        d = d1 - d2; se = np.sqrt(np.nan_to_num(s1) ** 2 + np.nan_to_num(s2) ** 2)
    d = d[mask]; se = np.where(np.isfinite(se[mask]), se[mask], 0.0)
    ok = np.isfinite(d); d = d[ok]; se = se[ok]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.empty(nb)
    for b in range(nb):
        idx = rng.randint(0, len(d), len(d))
        v[b] = (d[idx] - rng.normal(0.0, 1.0, len(idx)) * se[idx]).mean()
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

run.log("\n" + "=" * 100)
run.log(f"【M-D】 관문 (정확 pre × 기저선 {BW:.2f} · 개체 {NJ} · MIN_CELL {MIN_CELL_S})")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<5}{v}  {d}")

# ── M1 항등식
m1, lo1, hi1, n1_ = boot_diff(JM["f1"], None, SEED0 + 1, mask=JOK, const=0.0)
DIFF["M1"] = dict(mean=m1, lo=lo1, hi=hi1, n=n1_)
g_("M1", decide(lo1, hi1, EXACT_TIE, "<"),
   f"f1ₘ **{m1:.4f}** [{lo1:.4f}, {hi1:.4f}] vs 상한 {EXACT_TIE}   ← 정확 pre 항등식")

# ── M2 ★ 전제
m2, lo2, hi2, n2_ = boot_diff(JM["f2_16"], None, SEED0 + 2, mask=JOK, const=0.0)
DIFF["M2"] = dict(mean=m2, lo=lo2, hi=hi2, n=n2_)
g_("M2", decide(lo2, hi2, F2_COLLAPSE, "<"),
   f"★ f2_16ₘ **{m2:.4f}** [{lo2:.4f}, {hi2:.4f}] vs 상한 {F2_COLLAPSE}"
   f"   ← **두 축을 처음으로 동시에** (Q7-I‴ 0.6688 · Q7-K 0.5445)")
SEALED = not VERD["M2"].startswith("✅")
if SEALED:
    run.log("    ⛔ **상대 조기성이 아직 안 죽었다 — 아래 「수준」을 인용하지 않는다.**")
    run.log("       ★ 단 **M3a·M3b(차이)** 와 **대역별 반응 곡선**은 공통 오염이 상쇄되므로")
    run.log("       **봉인 아래에서도 읽는다**(R27 ②). 그래서 차이를 1차 지표로 사전등록했다")

# ── M3a ★ 주지표 — P 창이 음성 대조 창을 이기는가
m3, lo3, hi3, n3_ = boot_excess_diff("p_late", "stt", SEED0 + 3, JOK, q=BONF3 * 100)
DIFF["M3a"] = dict(mean=m3, lo=lo3, hi=hi3, n=n3_)
g_("M3a", decide(lo3, hi3, 0.0, ">"),
   f"★ (P_late 초과) − (STT 초과) **{m3:+.4f}** [{lo3:+.4f}, {hi3:+.4f}] · Bonf 3 · n={n3_}"
   f"   ← **미결/기각이면 「P 파 고유의 신호는 없다」** (Q7-K 는 +0.0279 미결)")
for nm_ in ("p_full", "p_early"):
    mx_, lx_, hx_, _ = boot_excess_diff(nm_, "stt", SEED0 + 30, JOK)
    run.log(f"    (참고·미보정) ({nm_} 초과) − (STT 초과) {mx_:+.4f} [{lx_:+.4f}, {hx_:+.4f}]")

# ── M3b ★ 형태가 보상성 휴지를 이기는가
m4, lo4, hi4, n4_ = boot_excess_diff("stt", "f3", SEED0 + 4, JOK, q=BONF3 * 100)
DIFF["M3b"] = dict(mean=m4, lo=lo4, hi=hi4, n=n4_)
g_("M3b", decide(lo4, hi4, 0.0, ">"),
   f"★ (STT 초과) − (f3 초과) **{m4:+.4f}** [{lo4:+.4f}, {hi4:+.4f}] · Bonf 3 · n={n4_}"
   f"   ← Q7-K 는 +0.1867 ✅")
for nm_ in ("stt", "stt_safe", "p_late", "f3", "f3_glob"):
    mx_, lx_, hx_, _ = boot_excess_diff(nm_, None, SEED0 + 40, JOK)
    run.log(f"    (수준·미보정) {nm_:<9} 초과 {mx_:+.4f} [{lx_:+.4f}, {hx_:+.4f}]"
            + ("   ⛔ 봉인" if SEALED else ""))

# ── M5 f3 붕괴의 해석을 반증할 수 있나
gl = [np.nanmean(CURVE[b]["M"]["f3_glob"][CURVE[b]["ok"]]) for b in CURVE]
lo_ = [np.nanmean(CURVE[b]["M"]["f3"][CURVE[b]["ok"]]) for b in CURVE]
run.log(f"  M5   (반증 시험) 대역별 f3 {['%.4f' % v for v in lo_]}")
run.log(f"       (반증 시험) 대역별 f3_glob {['%.4f' % v for v in gl]}")
run.log("       ▸ f3 만 단조로 내려가고 f3_glob 이 평평하면 「f3 붕괴는 국소 기저선 때문」이")
run.log("         맞다. **둘 다 내려가면 그 해석은 틀렸다** — 그때는 post 자체가 죽은 것이다")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF; CONFIG["sealed"] = bool(SEALED)
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【M-E】 ★ M4 — 3축 정합에서 STT 가 사라지나 (rate hysteresis)
# Q7-K L8 은 개체 6개라 판정 불가였다. MIN_CELL 을 1 로 내리고 pre 를 정확 정합으로
# 고정했으니 여기서 되찾은 표본으로 다시 묻는다. **이 퀘스트의 마지막 큰 미결**이다.
run.log("\n" + "=" * 100)
run.log(f"【M-E】 M4 — 3축 정합 (정확 pre × 기저선 {BW:.2f} × post {POST_BAND:.2f})")
run.log("=" * 100)
T2 = time.time()
A3 = {a: np.full(len(RS), np.nan) for a in ("stt", "stt_safe", "p_late", "f3")}
N3 = {a: np.full(len(RS), np.nan) for a in A3}
S3 = {a: np.full(len(RS), np.nan) for a in A3}
O3 = np.zeros(len(RS), bool); SF3 = np.full(len(RS), np.nan)
for i, r in enumerate(RS):
    tt = META[r]["tt"]; pre_m = META[r]["pre"]; med_ = float(np.median(pre_m))
    key3 = joint_key(pre_m, META[r]["lb"], BW * med_,
                     post_v=META[r]["post"], pbw=POST_BAND * med_)
    v, ks_, pr_, _, _, _ = cell_auc(SCORES[r]["f1"], tt, key3, MIN_CELL_S, MIN_CELL_N)
    if not (ks_ >= MIN_REC_S and pr_ >= MIN_REC_PAIR):
        continue
    O3[i] = True; SF3[i] = ks_ / max(META[r]["pos"], 1)
    for a in A3:
        A3[a][i] = cell_auc(SCORES[r][a], tt, key3, MIN_CELL_S, MIN_CELL_N)[0]
    acc = {a: [] for a in A3}
    for s_ in range(N_SHUF):
        rng = np.random.RandomState(SEED0 + 613 * (s_ + 1) + int(r))
        ts = rng.permutation(tt)
        Ss = build_scores(META[r]["X"], NAMES_G, ts, META[r]["MORPH"], SEED0 + 17 * (s_ + 1))
        if Ss is None:
            continue
        for a in A3:
            vv, _, pr2, _, _, _ = cell_auc(Ss[a], ts, key3, MIN_CELL_S, MIN_CELL_N)
            if pr2 >= 1:
                acc[a].append(vv)
    for a in A3:
        if len(acc[a]) >= 2:
            v_ = np.asarray(acc[a], float)
            N3[a][i] = float(v_.mean()); S3[a][i] = float(v_.std(ddof=1) / np.sqrt(len(v_)))
NJ3 = int(O3.sum())
run.log(f"  ({time.time()-T2:.0f}초)  3축 가능 **{NJ3}/{len(RS)}** 개체 · "
        f"남은 S 중앙 {np.nanmedian(SF3[O3]) if O3.any() else float('nan'):.3f}"
        f"   (Q7-K L8 은 6개체였다)")
if NJ3 < MIN_REC:
    g_("M4", "⛔ 측정 불가", f"3축 가능 개체 {NJ3} < {MIN_REC} — 표본이 없다")
else:
    for a in A3:
        run.log(f"    {a:<9} {np.nanmean(A3[a][O3]):.4f}  null {np.nanmean(N3[a][O3]):.4f}"
                f"  **초과 {np.nanmean(A3[a][O3]-N3[a][O3]):+.4f}**")
    d = A3["stt"] - N3["stt"]; se = np.nan_to_num(S3["stt"])
    dm = d[O3]; sm = se[O3]; okm = np.isfinite(dm)
    rng = np.random.RandomState(SEED0 + 5); dm = dm[okm]; sm = sm[okm]
    vv = np.array([(dm[ix] - rng.normal(0, 1, len(ix)) * sm[ix]).mean()
                   for ix in (rng.randint(0, len(dm), len(dm)) for _ in range(NB_BOOT))])
    m5_, lo5_, hi5_ = float(dm.mean()), float(np.percentile(vv, BONF3 * 100)), \
        float(np.percentile(vv, 100 - BONF3 * 100))
    DIFF["M4"] = dict(mean=m5_, lo=lo5_, hi=hi5_, n=int(len(dm)))
    g_("M4", decide(lo5_, hi5_, 0.0, ">"),
       f"★ 3축 STT 초과 **{m5_:+.4f}** [{lo5_:+.4f}, {hi5_:+.4f}] · Bonf 3 · n={len(dm)}"
       f"   ← **사라지면 STT 는 재분극 rate hysteresis** = 리듬의 다른 얼굴")
    run.log(f"    (2축 대비) 2축 STT 초과 {np.nanmean(JM['stt'][JOK]-NULL['stt'][JOK]):+.4f}"
            f" → 3축 {m5_:+.4f}   ← **같은 개체가 아니므로 세로 비교 주의**")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF; CONFIG["n_3axis"] = NJ3
run.save_json("config", CONFIG)

In [ ]:
# CELL 9 — 【M-F】 M6 — 강건성: 상쇄 검증 · LOO · 선택 편향 · 층화 간극
run.log("\n" + "=" * 100)
run.log("【M-F】 M6 — 강건성 (관문 아님)")
run.log("=" * 100)

# ── ① ★ 상쇄 검증 — 「차이는 봉인 밖」의 전제를 실측한다
#    잔여 조기성이 두 창을 똑같이 부풀린다면, 개체별 (차이)와 개체별 잔여는 무상관이어야.
dpe = (JM["p_late"] - NULL["p_late"]) - (JM["stt"] - NULL["stt"])
res = CURVE[BW]["g2"]
m_ = JOK & np.isfinite(dpe) & np.isfinite(res)
if int(m_.sum()) >= 5:
    rho = float(stats.spearmanr(dpe[m_], res[m_]).correlation)
    rng = np.random.RandomState(SEED0 + 61)
    bs = [stats.spearmanr(*(lambda ix: (dpe[m_][ix], res[m_][ix]))(
        rng.randint(0, int(m_.sum()), int(m_.sum())))).correlation for _ in range(600)]
    bs = np.array([b for b in bs if np.isfinite(b)])
    run.log(f"  ★ 상쇄 검증 — rho(개체별 (P_late−STT) 초과차, 개체별 f2 잔여) = "
            f"**{rho:+.3f}** [{np.percentile(bs,2.5):+.3f}, {np.percentile(bs,97.5):+.3f}] "
            f"· n={int(m_.sum())}")
    run.log("     ▸ 0 을 포함하면 **잔여 오염이 두 창을 같게 부풀린다**는 전제가 확인되고,")
    run.log("       M3a 를 **봉인 밖 결론**으로 쓸 수 있다(R27 ②)")
    CONFIG["cancel_rho"] = rho
else:
    run.log("  상쇄 검증 — 개체가 부족해 못 냈다")

# ── ② leave-one-out — 큰 개체 하나가 M3a·M3b 를 만들고 있나
run.log("\n  leave-one-out (개체 하나씩 빼고 다시)")
for nm_, a1, a2 in (("M3a", "p_late", "stt"), ("M3b", "stt", "f3")):
    base = DIFF[nm_]["mean"]
    idx = np.where(JOK)[0]
    vals = []
    for j in idx:
        mk = JOK.copy(); mk[j] = False
        vals.append(boot_excess_diff(a1, a2, SEED0 + 70, mk, nb=200)[0])
    vals = np.array(vals, float)
    k = int(np.argmax(np.abs(vals - base)))
    run.log(f"    {nm_}  기준 {base:+.4f} · LOO 범위 [{np.nanmin(vals):+.4f}, "
            f"{np.nanmax(vals):+.4f}] · 최대 영향 개체 **#{RS[idx[k]]}** ({vals[k]:+.4f})")
    flip = bool(np.nanmin(vals) * np.nanmax(vals) < 0)
    run.log("      ⚠️ **한 개체를 빼면 부호가 바뀐다**" if flip else "      ▸ 부호는 안정적이다")
    CONFIG.setdefault("loo", {})[nm_] = dict(base=base, lo=float(np.nanmin(vals)),
                                             hi=float(np.nanmax(vals)), flip=flip)

# ── ③ 선택 편향 (Q7-K L9 승계)
ISOF = np.array([META[r]["iso_frac"] for r in RS])
LAY = (("고립 S", ISOF >= ISO_HI), ("혼합", (ISOF < ISO_HI) & (ISOF > ISO_LO)),
       ("런 우세", ISOF <= ISO_LO))
PREV = np.array([META[r]["prev"] for r in RS])
RRSEP = np.array([abs(np.median(META[r]["pre"][~META[r]["tt"]])
                      - np.median(META[r]["pre"][META[r]["tt"]]))
                  / max(np.median(META[r]["pre"]), 1e-9) for r in RS])
run.log(f"\n  선택 편향 — 정합 가능 **{NJ}/{len(RS)}** (Q7-K 는 19/59)")
for nm_, msk in LAY:
    tot = int(msk.sum()); got = int((msk & JOK).sum()); g3 = int((msk & O3).sum())
    run.log(f"    {nm_:<8} 전체 {tot:>2} → 2축 **{got:>2}** · 3축 **{g3:>2}**"
            + ("   ⛔ **한 개도 안 남았다**" if tot > 0 and got == 0 else ""))
run.log(f"  RR 분리도 — 정합 {np.nanmedian(RRSEP[JOK]):.4f} vs 제외 "
        f"{np.nanmedian(RRSEP[~JOK]) if (~JOK).any() else float('nan'):.4f}"
        f"   |  유병률 — 정합 {np.nanmedian(PREV[JOK]):.4f} vs 제외 "
        f"{np.nanmedian(PREV[~JOK]) if (~JOK).any() else float('nan'):.4f}")
run.log("  ▸ 층이 0 에 가까우면 **결론의 적용 범위를 그 층 밖으로 넓히지 않는다**(사전등록)")

# ── ④ 층화 vs 매크로 간극 — 큰 개체가 끄는가
run.log("\n  비트 수준 층화 일치도 vs 매크로 (**매크로가 주지표** · 사전 선언)")
POOL = {}
for a in ("f1", "f2_16", "f3", "stt", "stt_safe", "p_late", "lr_all"):
    num = den = 0.0
    for i, r in enumerate(RS):
        if not JOK[i]:
            continue
        key = joint_key(META[r]["pre"], META[r]["lb"], BW * float(np.median(META[r]["pre"])))
        v, _, pr_, _, _, _ = cell_auc(SCORES[r][a], META[r]["tt"], key,
                                      MIN_CELL_S, MIN_CELL_N)
        if pr_ >= 1:
            num += v * pr_; den += pr_
    POOL[a] = num / den if den > 0 else float("nan")
    run.log(f"    {a:<9} 층화 {POOL[a]:.4f}  매크로 {np.nanmean(JM[a][JOK]):.4f}  "
            f"차 {POOL[a]-np.nanmean(JM[a][JOK]):+.4f}")
CONFIG["pooled"] = POOL
CONFIG["selection"] = dict(
    layers={nm_: dict(total=int(m.sum()), ax2=int((m & JOK).sum()),
                      ax3=int((m & O3).sum())) for nm_, m in LAY},
    rr_sep_matched=float(np.nanmedian(RRSEP[JOK])),
    prev_matched=float(np.nanmedian(PREV[JOK])))
run.save_json("config", CONFIG)

In [ ]:
# CELL 10 — 【M-G】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다(네모 방지).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

XS = [0.012 if b is None else b for b in CURVE]
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))
for a_, c_, lb in (("f1", "tab:red", "f1 (must be 0.5)"),
                   ("f2_16", "tab:purple", "f2_16 (must fall)"),
                   ("f3", "tab:green", "f3"), ("f3_glob", "tab:olive", "f3_glob"),
                   ("stt", "tab:orange", "STT"), ("p_late", "tab:blue", "P_late")):
    ax[0].plot(XS, [np.nanmean(CURVE[b]["M"][a_][CURVE[b]["ok"]]) for b in CURVE],
               "o-", color=c_, label=lb)
ax[0].axhline(0.5, ls="--", lw=0.8, color="crimson")
ax[0].axhline(F2_COLLAPSE, ls=":", lw=0.8, color="purple")
ax[0].set_xlabel("local-baseline band  (0.012 = exact-pre only)  <- narrower")
ax[0].set_ylabel("matched macro AUROC  (pre always exact)")
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

ax[1].plot(XS, [np.nanmedian(CURVE[b]["g2"][CURVE[b]["ok"]]) for b in CURVE],
           "s-", color="purple", label="f2 residual")
ax[1].plot(XS, [np.nanmedian(CURVE[b]["g1"][CURVE[b]["ok"]]) for b in CURVE],
           "s-", color="crimson", label="f1 residual (0 by design)")
ax2 = ax[1].twinx()
ax2.plot(XS, [int(CURVE[b]["ok"].sum()) for b in CURVE], "^--", color="gray",
         label="records")
ax2.set_ylabel("records matchable")
ax[1].set_yscale("symlog", linthresh=1e-4)
ax[1].set_xlabel("local-baseline band  <- narrower")
ax[1].set_ylabel("PAIRWISE residual"); ax[1].legend(fontsize=7, loc="center left")
ax[1].grid(alpha=.3)

aa = [a for a in ARMS if a not in ("f2_8",)]
ex_ = [np.nanmean(JM[a][JOK]) - np.nanmean(NULL[a][JOK]) for a in aa]
ax[2].barh(range(len(aa)), ex_, color=["tab:green" if v > 0 else "tab:red" for v in ex_])
ax[2].set_yticks(range(len(aa))); ax[2].set_yticklabels(aa, fontsize=7)
ax[2].axvline(0, color="k", lw=.8)
ax[2].set_xlabel("excess over own label-shuffle null (joint match)")
ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q7m_recover_power", fig)
plt.close(fig)
display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("관문 요약")
run.log("=" * 100)
for k in ("M1", "M2", "M3a", "M3b", "M4"):
    run.log(f"  {k:<5}{VERD.get(k, '(미실행)')}")
run.log("")
ok_ = lambda k: VERD.get(k, "").startswith("✅")
if not ok_("M1"):
    run.log("  ⛔ **정확 pre 인데 f1 이 0.5 가 아니다** — 정합기부터 고친다. 아래를 안 읽는다")
elif SEALED:
    run.log("  ⛔ **M2 미결 — 상대 조기성을 여전히 못 죽였다.** 수준은 인용하지 않는다.")
    run.log("     `MIN_CELL=1` 로 표본을 되찾고도 안 되면 **SVDB 로는 여기까지**다 —")
    run.log("     결론은 「형태가 없다」가 아니라 **「이 데이터로는 통제할 수 없다」**다.")
    run.log("     ★ 단 **M3a·M3b 의 차이**는 상쇄 검증(【M-F】①)이 받쳐주면 읽는다")
else:
    run.log("  ★ **두 축을 동시에 통제했다 — 이 퀘스트에서 처음이다.**")
    if ok_("M4"):
        run.log("     그리고 **post 까지 맞춰도 STT 가 남았다** → **형태의 독립 기여 확정**")
        run.log("     → Q7-G(교차환자 템플릿) 재개 · Q7-F′ 는 M3a 결과에 따라 판단")
    elif "M4" in VERD:
        run.log("     그런데 **post 를 맞추면 STT 가 사라진다** → STT 는 **재분극 rate")
        run.log("     hysteresis** = 리듬의 다른 얼굴이다. **형태 축 종료**")
if not ok_("M3a"):
    run.log("  ★ **P 창이 음성 대조 창을 못 이긴다** → 「P 파 고유의 신호는 없다」")
    run.log("     Q7-F(F2·F3) · Q7-K(L7) 에 이은 **세 번째 독립 확인**")

run.finish({
    "exp_id": "quest46_q7m_recover_power",
    "metric": "svdb_recover_stt_excess",
    "value": float(np.nanmean(JM["stt"][JOK]) - np.nanmean(NULL["stt"][JOK])),
    "passed": bool(ok_("M1") and ok_("M2")),
    "summary": ("정확 pre 고정 + 국소기저선 대역 + MIN_CELL 1 로 표본을 되찾고, "
                "차이를 1차 지표로 형태·post_rr 의 독립 기여를 물었다."),
    "verdicts": VERD, "diffs": DIFF, "band_selected": BW, "sealed": bool(SEALED),
    "audit_cell": CONFIG.get("audit_cell", {}), "curve": CONFIG.get("curve", {}),
    "null": CONFIG.get("null", {}), "pooled": CONFIG.get("pooled", {}),
    "selection": CONFIG.get("selection", {}), "loo": CONFIG.get("loo", {}),
    "cancel_rho": CONFIG.get("cancel_rho"), "n_scored": len(RS),
    "n_joint": NJ, "n_3axis": CONFIG.get("n_3axis"), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-recover-power`")